# Assignment 1:  Sentiment with Deep Neural Networks

Welcome to the first assignment of course 3. In this assignment, you will explore sentiment analysis using deep neural networks. 
## Outline
- [Part 1:  Import libraries and try out Trax](#1)
- [Part 2:  Importing the data](#2)
    - [2.1  Loading in the data](#2.1)
    - [2.2  Building the vocabulary](#2.2)
    - [2.3  Converting a tweet to a tensor](#2.3)
        - [Exercise 01](#ex01)
    - [2.4  Creating a batch generator](#2.4)
        - [Exercise 02](#ex02)
- [Part 3:  Defining classes](#3)
    - [3.1  ReLU class](#3.1)
        - [Exercise 03](#ex03)
    - [3.2  Dense class ](#3.2)
        - [Exercise 04](#ex04)
    - [3.3  Model](#3.3)
        - [Exercise 05](#ex05)
- [Part 4:  Training](#4)
    - [4.1  Training the model](#4.1)
        - [Exercise 06](#ex06)
    - [4.2  Practice Making a prediction](#4.2)
- [Part 5:  Evaluation  ](#5)
    - [5.1  Computing the accuracy on a batch](#5.1)
        - [Exercise 07](#ex07)
    - [5.2  Testing your model on Validation Data](#5.2)
        - [Exercise 08](#ex08)
- [Part 6:  Testing with your own input](#6)


In course 1, you implemented Logistic regression and Naive Bayes for sentiment analysis. However if you were to give your old models an example like:

<center> <span style='color:blue'> <b>This movie was almost good.</b> </span> </center>

Your model would have predicted a positive sentiment for that review. However, that sentence has a negative sentiment and indicates that the movie was not good. To solve those kinds of misclassifications, you will write a program that uses deep neural networks to identify sentiment in text. By completing this assignment, you will: 

- Understand how you can build/design a model using layers
- Train a model using a training loop
- Use a binary cross-entropy loss function
- Compute the accuracy of your model
- Predict using your own input

As you can tell, this model follows a similar structure to the one you previously implemented in the second course of this specialization. 
- Indeed most of the deep nets you will be implementing will have a similar structure. The only thing that changes is the model architecture, the inputs, and the outputs. Before starting the assignment, we will introduce you to the Google library `trax` that we use for building and training models.


Now we will show you how to compute the gradient of a certain function `f` by just using `  .grad(f)`. 

- Trax source code can be found on Github: [Trax](https://github.com/google/trax)
- The Trax code also uses the JAX library: [JAX](https://jax.readthedocs.io/en/latest/index.html)

<a name="1"></a>
# Part 1:  Import libraries and try out Trax

- Let's import libraries and look at an example of using the Trax library.

In [1]:
import os 
import random as rnd

# import relevant libraries
import trax

# set random seeds to make this notebook easier to replicate
trax.supervised.trainer_lib.init_random_number_generators(31)

# import trax.fastmath.numpy
import trax.fastmath.numpy as np

# import trax.layers
from trax import layers as tl

# import Layer from the utils.py file
from utils import Layer, load_tweets, process_tweet
#from utils import 


Error in cell 3: module 'trax.supervised.trainer_lib' has no attribute 'init_random_number_generators'


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 8, in <module>
AttributeError: module 'trax.supervised.trainer_lib' has no attribute 'init_random_number_generators'


In [2]:
# Create an array using trax.fastmath.numpy
a = np.array(5.0)

# View the returned array
display(a)

print(type(a))

Error in cell 4: name 'np' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 2, in <module>
NameError: name 'np' is not defined


Notice that trax.fastmath.numpy returns a DeviceArray from the jax library.

In [3]:
# Define a function that will use the trax.fastmath.numpy array
def f(x):
    
    # f = x^2
    return (x**2)

In [4]:
# Call the function
print(f"f(a) for a={a} is {f(a)}")

Error in cell 7: name 'a' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 2, in <module>
NameError: name 'a' is not defined


The gradient (derivative) of function `f` with respect to its input `x` is the derivative of $x^2$.
- The derivative of $x^2$ is $2x$.  
- When x is 5, then $2x=10$.

You can calculate the gradient of a function by using `trax.fastmath.grad(fun=)` and passing in the name of the function.
- In this case the function you want to take the gradient of is `f`.
- The object returned (saved in `grad_f` in this example) is a function that can calculate the gradient of f for a given trax.fastmath.numpy array.

In [5]:
# Directly use trax.fastmath.grad to calculate the gradient (derivative) of the function
grad_f = trax.fastmath.grad(fun=f)  # df / dx - Gradient of function f(x) with respect to x

# View the type of the retuned object (it's a function)
type(grad_f)

In [6]:
# Call the newly created function and pass in a value for x (the DeviceArray stored in 'a')
grad_calculation = grad_f(a)

# View the result of calling the grad_f function
display(grad_calculation)

Error in cell 10: name 'a' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 2, in <module>
NameError: name 'a' is not defined


The function returned by trax.fastmath.grad takes in x=5 and calculates the gradient of f, which is 2*x, which is 10. The value is also stored as a DeviceArray from the jax library.

<a name="2"></a>
# Part 2:  Importing the data

<a name="2.1"></a>
## 2.1  Loading in the data

Import the data set.  
- You may recognize this from earlier assignments in the specialization.
- Details of process_tweet function are available in utils.py file

In [7]:
## DO NOT EDIT THIS CELL

# Import functions from the utils.py file

import numpy as np

# Load positive and negative tweets
all_positive_tweets, all_negative_tweets = load_tweets()

# View the total number of positive and negative tweets.
print(f"The number of positive tweets: {len(all_positive_tweets)}")
print(f"The number of negative tweets: {len(all_negative_tweets)}")

# Split positive set into validation and training
val_pos   = all_positive_tweets[4000:] # generating validation set for positive tweets
train_pos  = all_positive_tweets[:4000]# generating training set for positive tweets

# Split negative set into validation and training
val_neg   = all_negative_tweets[4000:] # generating validation set for negative tweets
train_neg  = all_negative_tweets[:4000] # generating training set for nagative tweets

# Combine training data into one set
train_x = train_pos + train_neg 

# Combine validation data into one set
val_x  = val_pos + val_neg

# Set the labels for the training set (1 for positive, 0 for negative)
train_y = np.append(np.ones(len(train_pos)), np.zeros(len(train_neg)))

# Set the labels for the validation set (1 for positive, 0 for negative)
val_y  = np.append(np.ones(len(val_pos)), np.zeros(len(val_neg)))

print(f"length of train_x {len(train_x)}")
print(f"length of val_x {len(val_x)}")

Error in cell 13: name 'load_tweets' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 8, in <module>
NameError: name 'load_tweets' is not defined


Now import a function that processes tweets (we've provided this in the utils.py file).
- `process_tweets' removes unwanted characters e.g. hashtag, hyperlinks, stock tickers from tweet.
- It also returns a list of words (it tokenizes the original string).

In [8]:
# Import a function that processes the tweets
# from utils import process_tweet

# Try out function that processes tweets
print("original tweet at training position 0")
print(train_pos[0])

print("Tweet at training position 0 after processing:")
process_tweet(train_pos[0])

original tweet at training position 0
Error in cell 15: name 'train_pos' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 6, in <module>
NameError: name 'train_pos' is not defined


Notice that the function `process_tweet` keeps key words, removes the hash # symbol, and ignores usernames (words that begin with '@').  It also returns a list of the words.

<a name="2.2"></a>
## 2.2  Building the vocabulary

Now build the vocabulary.
- Map each word in each tweet to an integer (an "index"). 
- The following code does this for you, but please read it and understand what it's doing.
- Note that you will build the vocabulary based on the training data. 
- To do so, you will assign an index to everyword by iterating over your training set.

The vocabulary will also include some special tokens
- `__PAD__`: padding
- `</e>`: end of line
- `__UNK__`: a token representing any word that is not in the vocabulary.

In [9]:
# Build the vocabulary
# Unit Test Note - There is no test set here only train/val

# Include special tokens 
# started with pad, end of line and unk tokens
Vocab = {'__PAD__': 0, '__</e>__': 1, '__UNK__': 2} 

# Note that we build vocab using training data
for tweet in train_x: 
    processed_tweet = process_tweet(tweet)
    for word in processed_tweet:
        if word not in Vocab: 
            Vocab[word] = len(Vocab)
    
print("Total words in vocab are",len(Vocab))
display(Vocab)

Error in cell 18: name 'train_x' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 9, in <module>
NameError: name 'train_x' is not defined


The dictionary `Vocab` will look like this:
```CPP
{'__PAD__': 0,
 '__</e>__': 1,
 '__UNK__': 2,
 'followfriday': 3,
 'top': 4,
 'engag': 5,
 ...
```

- Each unique word has a unique integer associated with it.
- The total number of words in Vocab: 9088

<a name="2.3"></a>
## 2.3  Converting a tweet to a tensor

Write a function that will convert each tweet to a tensor (a list of unique integer IDs representing the processed tweet).
- Note, the returned data type will be a **regular Python `list()`**
    - You won't use TensorFlow in this function
    - You also won't use a numpy array
    - You also won't use trax.fastmath.numpy array
- For words in the tweet that are not in the vocabulary, set them to the unique ID for the token `__UNK__`.

##### Example
Input a tweet:
```CPP
'@happypuppy, is Maria happy?'
```

The tweet_to_tensor will first conver the tweet into a list of tokens (including only relevant words)
```CPP
['maria', 'happi']
```

Then it will convert each word into its unique integer

```CPP
[2, 56]
```
- Notice that the word "maria" is not in the vocabulary, so it is assigned the unique integer associated with the `__UNK__` token, because it is considered "unknown."



<a name="ex01"></a>
### Exercise 01
**Instructions:** Write a program `tweet_to_tensor` that takes in a tweet and converts it to an array of numbers. You can use the `Vocab` dictionary you just found to help create the tensor. 

- Use the vocab_dict parameter and not a global variable.
- Do not hard code the integer value for the `__UNK__` token.

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li>Map each word in tweet to corresponding token in 'Vocab'</li>
    <li>Use Python's Dictionary.get(key,value) so that the function returns a default value if the key is not found in the dictionary.</li>
</ul>
</p>


In [10]:
# UNQ_C1 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: tweet_to_tensor
def tweet_to_tensor(tweet, vocab_dict, unk_token='__UNK__', verbose=False):
    word_l = process_tweet(tweet)
    tensor_l = []
    unk_token_id = vocab_dict[unk_token]
    for word in word_l:
        word_id = vocab_dict.get(word, unk_token_id)
        tensor_l.append(word_id)
    return tensor_l


In [11]:
print("Actual tweet is\n", val_pos[0])
print("\nTensor of tweet:\n", tweet_to_tensor(val_pos[0], vocab_dict=Vocab))

Error in cell 24: name 'val_pos' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
NameError: name 'val_pos' is not defined


##### Expected output

```CPP
Actual tweet is
 Bro:U wan cut hair anot,ur hair long Liao bo
Me:since ord liao,take it easy lor treat as save $ leave it longer :)
Bro:LOL Sibei xialan

Tensor of tweet:
 [1065, 136, 479, 2351, 745, 8148, 1123, 745, 53, 2, 2672, 791, 2, 2, 349, 601, 2, 3489, 1017, 597, 4559, 9, 1065, 157, 2, 2]
```

In [12]:
# test tweet_to_tensor
def test_tweet_to_tensor():
    test_cases = [
        {
            'name': 'simple_test_check',
            'input': [val_pos[1], Vocab],
            'expected': tweet_to_tensor(val_pos[1], Vocab),
            'error': 'The function gives bad output for val_pos[1]. Test failed'
        },
        {
            'name': 'datatype_check',
            'input': [val_pos[1], Vocab],
            'expected': type([]),
            'error': 'Datatype mismatch. Need only list not np.array'
        },
        {
            'name': 'without_unk_check',
            'input': [val_pos[1], Vocab],
            'expected': None,
            'error': 'Unk word check not done- Please check if you included mapping for unknown word'
        }
    ]
    count = 0
    for test_case in test_cases:
        try:
            if test_case['name'] == 'simple_test_check':
                assert test_case['expected'] == tweet_to_tensor(*test_case['input'])
                count += 1
            if test_case['name'] == 'datatype_check':
                assert isinstance(tweet_to_tensor(*test_case['input']), test_case['expected'])
                count += 1
            if test_case['name'] == 'without_unk_check':
                assert None not in tweet_to_tensor(*test_case['input'])
                count += 1
        except:
            print(test_case['error'])
    if count == 3:
        print('\033[92m All tests passed')
    else:
        print(count, ' Tests passed out of 3')
test_tweet_to_tensor()


Error in cell 26: name 'val_pos' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 41, in <module>
  File "<string>", line 6, in test_tweet_to_tensor
NameError: name 'val_pos' is not defined


<a name="2.4"></a>
## 2.4  Creating a batch generator

Most of the time in Natural Language Processing, and AI in general we use batches when training our data sets. 
- If instead of training with batches of examples, you were to train a model with one example at a time, it would take a very long time to train the model. 
- You will now build a data generator that takes in the positive/negative tweets and returns a batch of training examples. It returns the model inputs, the targets (positive or negative labels) and the weight for each target (ex: this allows us to can treat some examples as more important to get right than others, but commonly this will all be 1.0). 

Once you create the generator, you could include it in a for loop

```CPP
for batch_inputs, batch_targets, batch_example_weights in data_generator:
    ...
```

You can also get a single batch like this:

```CPP
batch_inputs, batch_targets, batch_example_weights = next(data_generator)
```
The generator returns the next batch each time it's called. 
- This generator returns the data in a format (tensors) that you could directly use in your model.
- It returns a triple: the inputs, targets, and loss weights:
-- Inputs is a tensor that contains the batch of tweets we put into the model.
-- Targets is the corresponding batch of labels that we train to generate.
-- Loss weights here are just 1s with same shape as targets. Next week, you will use it to mask input padding.

<a name="ex02"></a>
### Exercise 02
Implement `data_generator`.

In [13]:
# UNQ_C2 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED: Data generator
def data_generator(data_pos, data_neg, batch_size, loop, vocab_dict, shuffle=False):
    assert batch_size % 2 == 0
    n_to_take = batch_size // 2
    pos_index = 0
    neg_index = 0
    len_pos = len(data_pos)
    len_neg = len(data_neg)
    pos_index_lines = list(range(len_pos))
    neg_index_lines = list(range(len_neg))
    if shuffle:
        rnd.shuffle(pos_index_lines)
        rnd.shuffle(neg_index_lines)
    stop = False
    while not stop:
        batch = []
        for i in range(n_to_take):
            if pos_index >= len_pos:
                if not loop:
                    stop = True
                    break
                pos_index = 0
                if shuffle:
                    rnd.shuffle(pos_index_lines)
            tweet = data_pos[pos_index_lines[pos_index]]
            tensor = tweet_to_tensor(tweet, vocab_dict)
            batch.append(tensor)
            pos_index += 1
        for i in range(n_to_take):
            if neg_index >= len_neg:
                if not loop:
                    stop = True
                    break
                neg_index = 0
                if shuffle:
                    rnd.shuffle(neg_index_lines)
            tweet = data_neg[neg_index_lines[neg_index]]
            tensor = tweet_to_tensor(tweet, vocab_dict)
            batch.append(tensor)
            neg_index += 1
        if stop:
            break
        max_len = max([len(t) for t in batch])
        max_len = 2**int(np.ceil(np.log2(max_len)))
        inputs = []
        targets = []
        for i in range(batch_size):
            tensor = batch[i]
            pad = [vocab_dict['__PAD__']] * (max_len - len(tensor))
            inputs.append(tensor + pad)
            if i < n_to_take:
                targets.append(1)
            else:
                targets.append(0)
        targets = np.array(targets)
        inputs = np.array(inputs)
        example_weights = np.ones_like(targets)
        yield inputs, targets, example_weights


Now you can use your data generator to create a data generator for the training data, and another data generator for the validation data.

We will create a third data generator that does not loop, for testing the final accuracy of the model.

In [14]:
# Set the random number generator for the shuffle procedure
rnd.seed(30) 

# Create the training data generator
def train_generator(batch_size, shuffle = False):
    return data_generator(train_pos, train_neg, batch_size, True, Vocab, shuffle)

# Create the validation data generator
def val_generator(batch_size, shuffle = False):
    return data_generator(val_pos, val_neg, batch_size, True, Vocab, shuffle)

# Create the validation data generator
def test_generator(batch_size, shuffle = False):
    return data_generator(val_pos, val_neg, batch_size, False, Vocab, shuffle)

# Get a batch from the train_generator and inspect.
inputs, targets, example_weights = next(train_generator(4, shuffle=True))

# this will print a list of 4 tensors padded with zeros
print(f'Inputs: {inputs}')
print(f'Targets: {targets}')
print(f'Example Weights: {example_weights}')

Error in cell 31: name 'train_pos' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 17, in <module>
  File "<string>", line 6, in train_generator
NameError: name 'train_pos' is not defined


In [15]:
# Test the train_generator

# Create a data generator for training data,
# which produces batches of size 4 (for tensors and their respective targets)
tmp_data_gen = train_generator(batch_size = 4)

# Call the data generator to get one batch and its targets
tmp_inputs, tmp_targets, tmp_example_weights = next(tmp_data_gen)

print(f"The inputs shape is {tmp_inputs.shape}")
print(f"The targets shape is {tmp_targets.shape}")
print(f"The example weights shape is {tmp_example_weights.shape}")

for i,t in enumerate(tmp_inputs):
    print(f"input tensor: {t}; target {tmp_targets[i]}; example weights {tmp_example_weights[i]}")

Error in cell 32: name 'train_pos' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 5, in <module>
  File "<string>", line 6, in train_generator
NameError: name 'train_pos' is not defined


##### Expected output

```CPP
The inputs shape is (4, 14)
The targets shape is (4,)
The example weights shape is (4,)
input tensor: [3 4 5 6 7 8 9 0 0 0 0 0 0 0]; target 1; example weights 1
input tensor: [10 11 12 13 14 15 16 17 18 19 20  9 21 22]; target 1; example weights 1
input tensor: [5738 2901 3761    0    0    0    0    0    0    0    0    0    0    0]; target 0; example weights 1
input tensor: [ 858  256 3652 5739  307 4458  567 1230 2767  328 1202 3761    0    0]; target 0; example weights 1
```

Now that you have your train/val generators, you can just call them and they will return tensors which correspond to your tweets in the first column and their corresponding labels in the second column. Now you can go ahead and start building your neural network. 

<a name="3"></a>
# Part 3:  Defining classes

In this part, you will write your own library of layers. It will be very similar
to the one used in Trax and also in Keras and PyTorch. Writing your own small
framework will help you understand how they all work and use them effectively
in the future.

Your framework will be based on the following `Layer` class from utils.py.

```CPP
class Layer(object):
    """ Base class for layers.
    """
      
    # Constructor
    def __init__(self):
        # set weights to None
        self.weights = None

    # The forward propagation should be implemented
    # by subclasses of this Layer class
    def forward(self, x):
        raise NotImplementedError

    # This function initializes the weights
    # based on the input signature and random key,
    # should be implemented by subclasses of this Layer class
    def init_weights_and_state(self, input_signature, random_key):
        pass

    # This initializes and returns the weights, do not override.
    def init(self, input_signature, random_key):
        self.init_weights_and_state(input_signature, random_key)
        return self.weights
 
    # __call__ allows an object of this class
    # to be called like it's a function.
    def __call__(self, x):
        # When this layer object is called, 
        # it calls its forward propagation function
        return self.forward(x)
```

<a name="3.1"></a>
## 3.1  ReLU class
You will now implement the ReLU activation function in a class below. The ReLU function looks as follows: 
<img src = "relu.jpg" style="width:300px;height:150px;"/>

$$ \mathrm{ReLU}(x) = \mathrm{max}(0,x) $$


<a name="ex03"></a>
### Exercise 03
**Instructions:** Implement the ReLU activation function below. Your function should take in a matrix or vector and it should transform all the negative numbers into 0 while keeping all the positive numbers intact. 

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Hints</b></font>
</summary>
<p>
<ul>
    <li>Please use numpy.maximum(A,k) to find the maximum between each element in A and a scalar k</li>
</ul>
</p>


In [16]:
# UNQ_C3 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: Relu
class Relu(Layer):
    def forward(self, x):
        activation = np.maximum(0, x)
        return activation


Error in cell 39: name 'Layer' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 3, in <module>
NameError: name 'Layer' is not defined


In [17]:
# Test your relu function
x = np.array([[-2.0, -1.0, 0.0], [0.0, 1.0, 2.0]], dtype=float)
relu_layer = Relu()
print("Test data is:")
print(x)
print("Output of Relu is:")
print(relu_layer(x))

Error in cell 40: name 'Relu' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 3, in <module>
NameError: name 'Relu' is not defined


##### Expected Outout
```CPP
Test data is:
[[-2. -1.  0.]
 [ 0.  1.  2.]]
Output of Relu is:
[[0. 0. 0.]
 [0. 1. 2.]]
```

<a name="3.2"></a>
## 3.2  Dense class 

### Exercise

Implement the forward function of the Dense class. 
- The forward function multiplies the input to the layer (`x`) by the weight matrix (`W`)

$$\mathrm{forward}(\mathbf{x},\mathbf{W}) = \mathbf{xW} $$

- You can use `numpy.dot` to perform the matrix multiplication.

Note that for more efficient code execution, you will use the trax version of `math`, which includes a trax version of `numpy` and also `random`.

Implement the weight initializer `new_weights` function
- Weights are initialized with a random key.
- The second parameter is a tuple for the desired shape of the weights (num_rows, num_cols)
- The num of rows for weights should equal the number of columns in x, because for forward propagation, you will multiply x times weights.

Please use `trax.fastmath.random.normal(key, shape, dtype=tf.float32)` to generate random values for the weight matrix. The key difference between this function
and the standard `numpy` randomness is the explicit use of random keys, which
need to be passed. While it can look tedious at the first sight to pass the random key everywhere, you will learn in Course 4 why this is very helpful when
implementing some advanced models.
- `key` can be generated by calling `random.get_prng(seed=)` and passing in a number for the `seed`.
- `shape` is a tuple with the desired shape of the weight matrix.
    - The number of rows in the weight matrix should equal the number of columns in the variable `x`.  Since `x` may have 2 dimensions if it reprsents a single training example (row, col), or three dimensions (batch_size, row, col), get the last dimension from the tuple that holds the dimensions of x.
    - The number of columns in the weight matrix is the number of units chosen for that dense layer.  Look at the `__init__` function to see which variable stores the number of units.
- `dtype` is the data type of the values in the generated matrix; keep the default of `tf.float32`. In this case, don't explicitly set the dtype (just let it use the default value).

Set the standard deviation of the random values to 0.1
- The values generated have a mean of 0 and standard deviation of 1.
- Set the default standard deviation `stdev` to be 0.1 by multiplying the standard deviation to each of the values in the weight matrix.

In [18]:
# use the fastmath module within trax
from trax import fastmath

# use the numpy module from trax
np = fastmath.numpy

# use the fastmath.random module from trax
random = fastmath.random

In [19]:
# See how the fastmath.trax.random.normal function works
tmp_key = random.get_prng(seed=1)
print("The random seed generated by random.get_prng")
display(tmp_key)

print("choose a matrix with 2 rows and 3 columns")
tmp_shape=(2,3)
display(tmp_shape)

# Generate a weight matrix
# Note that you'll get an error if you try to set dtype to tf.float32, where tf is tensorflow
# Just avoid setting the dtype and allow it to use the default data type
tmp_weight = trax.fastmath.random.normal(key=tmp_key, shape=tmp_shape)

print("Weight matrix generated with a normal distribution with mean 0 and stdev of 1")
display(tmp_weight)

The random seed generated by random.get_prng
[0 1]
choose a matrix with 2 rows and 3 columns
(2, 3)
Weight matrix generated with a normal distribution with mean 0 and stdev of 1
[[-0.15443718  0.08470728 -0.13598049]
 [-0.15503626  1.2666674   0.14829758]]


<a name="ex04"></a>
### Exercise 04

Implement the `Dense` class.

In [20]:
# UNQ_C4 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: Dense
class Dense(Layer):
    def __init__(self, n_units, init_stdev=0.1):
        self._n_units = n_units
        self._init_stdev = init_stdev
    def forward(self, x):
        dense = np.dot(x, self.weights)
        return dense
    def init_weights_and_state(self, input_signature, random_key):
        input_shape = input_signature.shape
        w = random.normal(key=random_key, shape=(input_shape[-1], self._n_units)) * self._init_stdev
        self.weights = w
        return self.weights, ()


Error in cell 46: name 'Layer' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 3, in <module>
NameError: name 'Layer' is not defined


In [21]:
# Testing your Dense layer 
dense_layer = Dense(n_units=10)  #sets  number of units in dense layer
random_key = random.get_prng(seed=0)  # sets random seed
z = np.array([[2.0, 7.0, 25.0]]) # input array 

dense_layer.init(z, random_key)
print("Weights are\n ",dense_layer.weights) #Returns randomly generated weights
print("Foward function output is ", dense_layer(z)) # Returns multiplied values of units and weights

Error in cell 47: name 'Dense' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 2, in <module>
NameError: name 'Dense' is not defined


##### Expected Outout
```CPP
Weights are
  [[-0.02837108  0.09368162 -0.10050076  0.14165013  0.10543301  0.09108126
  -0.04265672  0.0986188  -0.05575325  0.00153249]
 [-0.20785688  0.0554837   0.09142365  0.05744595  0.07227863  0.01210617
  -0.03237354  0.16234995  0.02450038 -0.13809784]
 [-0.06111237  0.01403724  0.08410042 -0.1094358  -0.10775021 -0.11396459
  -0.05933381 -0.01557652 -0.03832145 -0.11144515]]
Foward function output is  [[-3.0395496   0.9266802   2.5414743  -2.050473   -1.9769388  -2.582209
  -1.7952735   0.94427425 -0.8980402  -3.7497487 ]]
```

<a name="3.3"></a>
## 3.3  Model

Now you will implement a classifier using neural networks. Here is the model architecture you will be implementing. 

<img src = "nn.jpg" style="width:400px;height:250px;"/>

For the model implementation, you will use the Trax layers library `tl`.
Note that the second character of `tl` is the lowercase of letter `L`, not the number 1. Trax layers are very similar to the ones you implemented above,
but in addition to trainable weights also have a non-trainable state.
State is used in layers like batch normalization and for inference, you will learn more about it in course 4.

First, look at the code of the Trax Dense layer and compare to your implementation above.
- [tl.Dense](https://github.com/google/trax/blob/master/trax/layers/core.py#L29): Trax Dense layer implementation

One other important layer that you will use a lot is one that allows to execute one layer after another in sequence.
- [tl.Serial](https://github.com/google/trax/blob/master/trax/layers/combinators.py#L26): Combinator that applies layers serially.  
    - You can pass in the layers as arguments to `Serial`, separated by commas. 
    - For example: `tl.Serial(tl.Embeddings(...), tl.Mean(...), tl.Dense(...), tl.LogSoftmax(...))`

Please use the `help` function to view documentation for each layer.

In [22]:
# View documentation on tl.Dense
help(tl.Dense)

Error in cell 50: name 'tl' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 2, in <module>
NameError: name 'tl' is not defined


In [23]:
# View documentation on tl.Serial
help(tl.Serial)

Error in cell 51: name 'tl' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 2, in <module>
NameError: name 'tl' is not defined


- [tl.Embedding](https://github.com/google/trax/blob/1372b903bb66b0daccee19fd0b1fdf44f659330b/trax/layers/core.py#L113): Layer constructor function for an embedding layer.  
    - `tl.Embedding(vocab_size, d_feature)`.
    - `vocab_size` is the number of unique words in the given vocabulary.
    - `d_feature` is the number of elements in the word embedding (some choices for a word embedding size range from 150 to 300, for example).

In [24]:
# View documentation for tl.Embedding
help(tl.Embedding)

Error in cell 53: name 'tl' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 2, in <module>
NameError: name 'tl' is not defined


In [25]:
tmp_embed = tl.Embedding(vocab_size=3, d_feature=2)
display(tmp_embed)

Error in cell 54: name 'tl' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
NameError: name 'tl' is not defined


- [tl.Mean](https://github.com/google/trax/blob/1372b903bb66b0daccee19fd0b1fdf44f659330b/trax/layers/core.py#L276): Calculates means across an axis.  In this case, please choose axis = 1 to get an average embedding vector (an embedding vector that is an average of all words in the vocabulary).  
- For example, if the embedding matrix is 300 elements and vocab size is 10,000 words, taking the mean of the embedding matrix along axis=1 will yield a vector of 300 elements.

In [26]:
# view the documentation for tl.mean
help(tl.Mean)

Error in cell 56: name 'tl' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 2, in <module>
NameError: name 'tl' is not defined


In [27]:
# Pretend the embedding matrix uses 
# 2 elements for embedding the meaning of a word
# and has a vocabulary size of 3
# So it has shape (2,3)
tmp_embed = np.array([[1,2,3,],
                    [4,5,6]
                   ])

# take the mean along axis 0
print("The mean along axis 0 creates a vector whose length equals the vocabulary size")
display(np.mean(tmp_embed,axis=0))

print("The mean along axis 1 creates a vector whose length equals the number of elements in a word embedding")
display(np.mean(tmp_embed,axis=1))

The mean along axis 0 creates a vector whose length equals the vocabulary size
[2.5 3.5 4.5]
The mean along axis 1 creates a vector whose length equals the number of elements in a word embedding
[2. 5.]


- [tl.LogSoftmax](https://github.com/google/trax/blob/1372b903bb66b0daccee19fd0b1fdf44f659330b/trax/layers/core.py#L242): Implements log softmax function
- Here, you don't need to set any parameters for `LogSoftMax()`.

In [28]:
help(tl.LogSoftmax)

Error in cell 59: name 'tl' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
NameError: name 'tl' is not defined


**Online documentation**

- [tl.Dense](https://trax-ml.readthedocs.io/en/latest/trax.layers.html#trax.layers.core.Dense)

- [tl.Serial](https://trax-ml.readthedocs.io/en/latest/trax.layers.html#module-trax.layers.combinators)

- [tl.Embedding](https://trax-ml.readthedocs.io/en/latest/trax.layers.html#trax.layers.core.Embedding)

- [tl.Mean](https://trax-ml.readthedocs.io/en/latest/trax.layers.html#trax.layers.core.Mean)

- [tl.LogSoftmax](https://trax-ml.readthedocs.io/en/latest/trax.layers.html#trax.layers.core.LogSoftmax)

<a name="ex05"></a>
### Exercise 05
Implement the classifier function. 

In [29]:
# UNQ_C5 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: classifier
def classifier(vocab_size=len(Vocab), embedding_dim=256, output_dim=2, mode='train'):
    embed_layer = tl.Embedding(vocab_size=vocab_size, d_feature=embedding_dim)
    mean_layer = tl.Mean(axis=1)
    dense_output_layer = tl.Dense(n_units=output_dim)
    log_softmax_layer = tl.LogSoftmax()
    model = tl.Serial(embed_layer, mean_layer, dense_output_layer, log_softmax_layer)
    return model


In [30]:
tmp_model = classifier()

Error in cell 63: name 'tl' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
  File "<string>", line 4, in classifier
NameError: name 'tl' is not defined


In [31]:
print(type(tmp_model))
display(tmp_model)

Error in cell 64: name 'tmp_model' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
NameError: name 'tmp_model' is not defined


##### Expected Outout
```CPP
<class 'trax.layers.combinators.Serial'>
Serial[
  Embedding_9088_256
  Mean
  Dense_2
  LogSoftmax
]
```

<a name="4"></a>
# Part 4:  Training

To train a model on a task, Trax defines an abstraction [`trax.supervised.training.TrainTask`](https://trax-ml.readthedocs.io/en/latest/trax.supervised.html#trax.supervised.training.TrainTask) which packages the train data, loss and optimizer (among other things) together into an object.

Similarly to evaluate a model, Trax defines an abstraction [`trax.supervised.training.EvalTask`](https://trax-ml.readthedocs.io/en/latest/trax.supervised.html#trax.supervised.training.EvalTask) which packages the eval data and metrics (among other things) into another object.

The final piece tying things together is the [`trax.supervised.training.Loop`](https://trax-ml.readthedocs.io/en/latest/trax.supervised.html#trax.supervised.training.Loop) abstraction that is a very simple and flexible way to put everything together and train the model, all the while evaluating it and saving checkpoints.
Using `Loop` will save you a lot of code compared to always writing the training loop by hand, like you did in courses 1 and 2. More importantly, you are less likely to have a bug in that code that would ruin your training.

In [32]:

# View documentation for trax.supervised.training.TrainTask
help(trax.supervised.training.TrainTask)

Help on class TrainTask in module trax.supervised.training:

class TrainTask(builtins.object)
 |  TrainTask(
 |      labeled_data,
 |      loss_layer,
 |      optimizer,
 |      lr_schedule=None,
 |      n_steps_per_checkpoint=100,
 |      n_steps_per_permanent_checkpoint=None,
 |      loss_name=None,
 |      sample_batch=None,
 |      export_prefix=None
 |  )
 |
 |  A supervised task (labeled data + feedback mechanism) for training.
 |
 |  Methods defined here:
 |
 |  __init__(
 |      self,
 |      labeled_data,
 |      loss_layer,
 |      optimizer,
 |      lr_schedule=None,
 |      n_steps_per_checkpoint=100,
 |      n_steps_per_permanent_checkpoint=None,
 |      loss_name=None,
 |      sample_batch=None,
 |      export_prefix=None
 |  )
 |      Configures a training task.
 |
 |      Args:
 |        labeled_data: Iterator of batches of labeled data tuples. Each tuple has
 |            1+ data (input value) tensors followed by 1 label (target value)
 |            tensor.  All tensor

In [33]:
# View documentation for trax.supervised.training.EvalTask
help(trax.supervised.training.EvalTask)

Help on class EvalTask in module trax.supervised.training:

class EvalTask(builtins.object)
 |  EvalTask(
 |      labeled_data,
 |      metrics,
 |      metric_names=None,
 |      n_eval_batches=1,
 |      sample_batch=None,
 |      export_prefix=None
 |  )
 |
 |  Labeled data plus scalar functions for (periodically) measuring a model.
 |
 |  An eval task specifies how (``labeled_data`` + ``metrics``) and with what
 |  precision (``n_eval_batches``) to measure a model as it is training.
 |  The variance of each scalar output is reduced by measuring over multiple
 |  (``n_eval_batches``) batches and reporting the average from those
 |  measurements.
 |
 |  Methods defined here:
 |
 |  __init__(
 |      self,
 |      labeled_data,
 |      metrics,
 |      metric_names=None,
 |      n_eval_batches=1,
 |      sample_batch=None,
 |      export_prefix=None
 |  )
 |      Configures an eval task: named metrics run with a given data source.
 |
 |      Args:
 |        labeled_data: Iterator of b

In [34]:
# View documentation for trax.supervised.training.Loop
help(trax.supervised.training.Loop)

Help on class Loop in module trax.supervised.training:

class Loop(builtins.object)
 |  Loop(
 |      model,
 |      tasks,
 |      eval_model=None,
 |      eval_tasks=None,
 |      output_dir=None,
 |      checkpoint_at=None,
 |      checkpoint_low_metric=None,
 |      checkpoint_high_metric=None,
 |      permanent_checkpoint_at=None,
 |      eval_at=None,
 |      which_task=None,
 |      n_devices=None,
 |      random_seed=None,
 |      loss_chunk_size=0,
 |      use_memory_efficient_trainer=False,
 |      adasum=False,
 |      callbacks=None
 |  )
 |
 |  Loop that can run for a given number of steps to train a supervised model.
 |
 |  Can train the model on multiple tasks by interleaving updates according to the
 |  ``which_task`` argument.
 |
 |  The typical supervised training process randomly initializes a model and
 |  updates its weights via feedback (loss-derived gradients) from a training
 |  task, by looping through batches of labeled data. A training loop can also
 |  be co

In [35]:
# View optimizers that you could choose from
help(trax.optimizers)

Help on package trax.optimizers in trax:

NAME
    trax.optimizers - Optimizers for use with Trax layers.

PACKAGE CONTENTS
    adafactor
    adam
    base
    momentum
    optimizers_test
    rms_prop
    sm3
    trainer
    trainer_test

FUNCTIONS
    opt_configure(*args, **kwargs)

FILE
    c:\users\admin\appdata\local\programs\python\python313\lib\site-packages\trax\optimizers\__init__.py




Notice some available optimizers include:
```CPP
    adafactor
    adam
    momentum
    rms_prop
    sm3
```

<a name="4.1"></a>
## 4.1  Training the model

Now you are going to train your model. 

Let's define the `TrainTask`, `EvalTask` and `Loop` in preparation to train the model.

In [36]:
from trax.supervised import training

batch_size = 16
rnd.seed(271)

train_task = training.TrainTask(
    labeled_data=train_generator(batch_size=batch_size, shuffle=True),
    loss_layer=tl.CrossEntropyLoss(),
    optimizer=trax.optimizers.Adam(0.01),
    n_steps_per_checkpoint=10,
)

eval_task = training.EvalTask(
    labeled_data=val_generator(batch_size=batch_size, shuffle=True),
    metrics=[tl.CrossEntropyLoss(), tl.Accuracy()],
)

model = classifier()

Error in cell 73: name 'train_pos' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 7, in <module>
  File "<string>", line 6, in train_generator
NameError: name 'train_pos' is not defined


This defines a model trained using [`tl.CrossEntropyLoss`](https://trax-ml.readthedocs.io/en/latest/trax.layers.html#trax.layers.metrics.CrossEntropyLoss) optimized with the [`trax.optimizers.Adam`](https://trax-ml.readthedocs.io/en/latest/trax.optimizers.html#trax.optimizers.adam.Adam) optimizer, all the while tracking the accuracy using [`tl.Accuracy`](https://trax-ml.readthedocs.io/en/latest/trax.layers.html#trax.layers.metrics.Accuracy) metric. We also track `tl.CrossEntropyLoss` on the validation set.

Now let's make an output directory and train the model.

In [37]:
output_dir = '~/model/'
output_dir_expand = os.path.expanduser(output_dir)
print(output_dir_expand)

C:\Users\admin/model/


<a name="ex06"></a>
### Exercise 06
**Instructions:** Implement `train_model` to train the model (`classifier` that you wrote earlier) for the given number of training steps (`n_steps`) using `TrainTask`, `EvalTask` and `Loop`.

In [38]:
# UNQ_C6 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: train_model
def train_model(classifier, train_task, eval_task, n_steps, output_dir):
    training_loop = training.Loop(
        classifier,
        train_task,
        eval_tasks=[eval_task],
        output_dir=output_dir
    )
    training_loop.run(n_steps)
    return training_loop


In [39]:
training_loop = train_model(model, train_task, eval_task, 100, output_dir_expand)

Error in cell 79: name 'model' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
NameError: name 'model' is not defined


##### Expected output (Approximately)
```CPP
Step      1: train CrossEntropyLoss |  0.88939196
Step      1: eval  CrossEntropyLoss |  0.68833977
Step      1: eval          Accuracy |  0.50000000
Step     10: train CrossEntropyLoss |  0.61036736
Step     10: eval  CrossEntropyLoss |  0.52182281
Step     10: eval          Accuracy |  0.68750000
Step     20: train CrossEntropyLoss |  0.34137666
Step     20: eval  CrossEntropyLoss |  0.20654774
Step     20: eval          Accuracy |  1.00000000
Step     30: train CrossEntropyLoss |  0.20208922
Step     30: eval  CrossEntropyLoss |  0.21594886
Step     30: eval          Accuracy |  0.93750000
Step     40: train CrossEntropyLoss |  0.19611198
Step     40: eval  CrossEntropyLoss |  0.17582777
Step     40: eval          Accuracy |  1.00000000
Step     50: train CrossEntropyLoss |  0.11203773
Step     50: eval  CrossEntropyLoss |  0.07589275
Step     50: eval          Accuracy |  1.00000000
Step     60: train CrossEntropyLoss |  0.09375446
Step     60: eval  CrossEntropyLoss |  0.09290724
Step     60: eval          Accuracy |  1.00000000
Step     70: train CrossEntropyLoss |  0.08785903
Step     70: eval  CrossEntropyLoss |  0.09610598
Step     70: eval          Accuracy |  1.00000000
Step     80: train CrossEntropyLoss |  0.08858261
Step     80: eval  CrossEntropyLoss |  0.02319432
Step     80: eval          Accuracy |  1.00000000
Step     90: train CrossEntropyLoss |  0.05699894
Step     90: eval  CrossEntropyLoss |  0.01778970
Step     90: eval          Accuracy |  1.00000000
Step    100: train CrossEntropyLoss |  0.03663783
Step    100: eval  CrossEntropyLoss |  0.00210550
Step    100: eval          Accuracy |  1.00000000
```

<a name="4.2"></a>
## 4.2  Practice Making a prediction

Now that you have trained a model, you can access it as `training_loop.model` object. We will actually use `training_loop.eval_model` and in the next weeks you will learn why we sometimes use a different model for evaluation, e.g., one without dropout. For now, make predictions with your model.

Use the training data just to see how the prediction process works.  
- Later, you will use validation data to evaluate your model's performance.


In [40]:
# Create a generator object
tmp_train_generator = train_generator(16)

# get one batch
tmp_batch = next(tmp_train_generator)

# Position 0 has the model inputs (tweets as tensors)
# position 1 has the targets (the actual labels)
tmp_inputs, tmp_targets, tmp_example_weights = tmp_batch

print(f"The batch is a tuple of length {len(tmp_batch)} because position 0 contains the tweets, and position 1 contains the targets.") 
print(f"The shape of the tweet tensors is {tmp_inputs.shape} (num of examples, length of tweet tensors)")
print(f"The shape of the labels is {tmp_targets.shape}, which is the batch size.")
print(f"The shape of the example_weights is {tmp_example_weights.shape}, which is the same as inputs/targets size.")

Error in cell 82: name 'train_pos' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 2, in <module>
  File "<string>", line 6, in train_generator
NameError: name 'train_pos' is not defined


In [41]:
# feed the tweet tensors into the model to get a prediction
tmp_pred = training_loop.eval_model(tmp_inputs)
print(f"The prediction shape is {tmp_pred.shape}, num of tensor_tweets as rows")
print("Column 0 is the probability of a negative sentiment (class 0)")
print("Column 1 is the probability of a positive sentiment (class 1)")
print()
print("View the prediction array")
tmp_pred

Error in cell 83: name 'training_loop' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 2, in <module>
NameError: name 'training_loop' is not defined


To turn these probabilities into categories (negative or positive sentiment prediction), for each row:
- Compare the probabilities in each column.
- If column 1 has a value greater than column 0, classify that as a positive tweet.
- Otherwise if column 1 is less than or equal to column 0, classify that example as a negative tweet.

In [42]:
# turn probabilites into category predictions
tmp_is_positive = tmp_pred[:,1] > tmp_pred[:,0]
for i, p in enumerate(tmp_is_positive):
    print(f"Neg log prob {tmp_pred[i,0]:.4f}\tPos log prob {tmp_pred[i,1]:.4f}\t is positive? {p}\t actual {tmp_targets[i]}")

Error in cell 85: name 'tmp_pred' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 2, in <module>
NameError: name 'tmp_pred' is not defined. Did you mean: 'tmp_key'?


Notice that since you are making a prediction using a training batch, it's more likely that the model's predictions match the actual targets (labels).  
- Every prediction that the tweet is positive is also matching the actual target of 1 (positive sentiment).
- Similarly, all predictions that the sentiment is not positive matches the actual target of 0 (negative sentiment)

One more useful thing to know is how to compare if the prediction is matching the actual target (label).  
- The result of calculation `is_positive` is a boolean.
- The target is a type trax.fastmath.numpy.int32
- If you expect to be doing division, you may prefer to work with decimal numbers with the data type type trax.fastmath.numpy.int32

In [43]:
# View the array of booleans
print("Array of booleans")
display(tmp_is_positive)

# convert boolean to type int32
# True is converted to 1
# False is converted to 0
tmp_is_positive_int = tmp_is_positive.astype(np.int32)


# View the array of integers
print("Array of integers")
display(tmp_is_positive_int)

# convert boolean to type float32
tmp_is_positive_float = tmp_is_positive.astype(np.float32)

# View the array of floats
print("Array of floats")
display(tmp_is_positive_float)

Array of booleans
Error in cell 88: name 'tmp_is_positive' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 3, in <module>
NameError: name 'tmp_is_positive' is not defined


In [44]:
tmp_pred.shape

Error in cell 89: name 'tmp_pred' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
NameError: name 'tmp_pred' is not defined. Did you mean: 'tmp_key'?


Note that Python usually does type conversion for you when you compare a boolean to an integer
- True compared to 1 is True, otherwise any other integer is False.
- False compared to 0 is True, otherwise any ohter integer is False.

In [45]:
print(f"True == 1: {True == 1}")
print(f"True == 2: {True == 2}")
print(f"False == 0: {False == 0}")
print(f"False == 2: {False == 2}")

True == 1: True
True == 2: False
False == 0: True
False == 2: False


However, we recommend that you keep track of the data type of your variables to avoid unexpected outcomes.  So it helps to convert the booleans into integers
- Compare 1 to 1 rather than comparing True to 1.

Hopefully you are now familiar with what kinds of inputs and outputs the model uses when making a prediction.
- This will help you implement a function that estimates the accuracy of the model's predictions.

<a name="5"></a>
# Part 5:  Evaluation  

<a name="5.1"></a>
## 5.1  Computing the accuracy on a batch

You will now write a function that evaluates your model on the validation set and returns the accuracy. 
- `preds` contains the predictions.
    - Its dimensions are `(batch_size, output_dim)`.  `output_dim` is two in this case.  Column 0 contains the probability that the tweet belongs to class 0 (negative sentiment). Column 1 contains probability that it belongs to class 1 (positive sentiment).
    - If the probability in column 1 is greater than the probability in column 0, then interpret this as the model's prediction that the example has label 1 (positive sentiment).  
    - Otherwise, if the probabilities are equal or the probability in column 0 is higher, the model's prediction is 0 (negative sentiment).
- `y` contains the actual labels.
- `y_weights` contains the weights to give to predictions.

<a name="ex07"></a>
### Exercise 07
Implement `compute_accuracy`.

In [46]:
# UNQ_C7 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: compute_accuracy
def compute_accuracy(preds, y, y_weights):
    if len(preds.shape) > 1 and preds.shape[-1] > 1:
        preds = np.argmax(preds, axis=-1)
    is_correct = np.equal(preds, y)
    correct_weights = is_correct * y_weights
    weighted_num_correct = np.sum(correct_weights)
    sum_weights = np.sum(y_weights)
    accuracy = weighted_num_correct / sum_weights
    return accuracy, weighted_num_correct, sum_weights


In [47]:
# test your function
tmp_val_generator = val_generator(64)

# get one batch
tmp_batch = next(tmp_val_generator)

# Position 0 has the model inputs (tweets as tensors)
# position 1 has the targets (the actual labels)
tmp_inputs, tmp_targets, tmp_example_weights = tmp_batch

# feed the tweet tensors into the model to get a prediction
tmp_pred = training_loop.eval_model(tmp_inputs)

tmp_acc, tmp_num_correct, tmp_num_predictions = compute_accuracy(preds=tmp_pred, y=tmp_targets, y_weights=tmp_example_weights)

print(f"Model's prediction accuracy on a single training batch is: {100 * tmp_acc}%")
print(f"Weighted number of correct predictions {tmp_num_correct}; weighted number of total observations predicted {tmp_num_predictions}")

Error in cell 97: name 'val_pos' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 2, in <module>
  File "<string>", line 10, in val_generator
NameError: name 'val_pos' is not defined


##### Expected output (Approximately)

```
Model's prediction accuracy on a single training batch is: 100.0%
Weighted number of correct predictions 64.0; weighted number of total observations predicted 64
```

<a name="5.2"></a>
## 5.2  Testing your model on Validation Data

Now you will write test your model's prediction accuracy on validation data. 

This program will take in a data generator and your model. 
- The generator allows you to get batches of data. You can use it with a `for` loop:

```
for batch in iterator: 
   # do something with that batch
```

`batch` has dimensions `(X, Y, weights)`. 
- Column 0 corresponds to the tweet as a tensor (input).
- Column 1 corresponds to its target (actual label, positive or negative sentiment).
- Column 2 corresponds to the weights associated (example weights)
- You can feed the tweet into model and it will return the predictions for the batch. 


<a name="ex08"></a>
### Exercise 08

**Instructions:** 
- Compute the accuracy over all the batches in the validation iterator. 
- Make use of `compute_accuracy`, which you recently implemented, and return the overall accuracy.

In [48]:
# UNQ_C8 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: test_model
def test_model(generator, model):
    accuracy = 0.
    total_num_correct = 0
    total_num_pred = 0
    for batch in generator:
        inputs = batch[0]
        targets = batch[1]
        y_weights = batch[2]
        predictions = model(inputs)
        preds = np.argmax(predictions, axis=-1)
        acc, weighted_num_correct, sum_weights = compute_accuracy(preds, targets, y_weights)
        total_num_correct += weighted_num_correct
        total_num_pred += sum_weights
    accuracy = total_num_correct / total_num_pred
    return accuracy


In [49]:
# DO NOT EDIT THIS CELL
# testing the accuracy of your model: this takes around 20 seconds
model = training_loop.eval_model
accuracy = test_model(test_generator(16), model)

print(f'The accuracy of your model on the validation set is {accuracy:.4f}', )

Error in cell 102: name 'training_loop' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 3, in <module>
NameError: name 'training_loop' is not defined


##### Expected Output (Approximately)

```CPP
The accuracy of your model on the validation set is 0.9931
```

<a name="6"></a>
# Part 6:  Testing with your own input

Finally you will test with your own input. You will see that deepnets are more powerful than the older methods you have used before. Although you go close to 100% accuracy on the first two assignments, the task was way easier. 

In [50]:
# this is used to predict on your own sentnece
def predict(sentence):
    inputs = np.array(tweet_to_tensor(sentence, vocab_dict=Vocab))
    
    # Batch size 1, add dimension for batch, to work with the model
    inputs = inputs[None, :]  
    
    # predict with the model
    preds_probs = model(inputs)
    
    # Turn probabilities into categories
    preds = int(preds_probs[0, 1] > preds_probs[0, 0])
    
    sentiment = "negative"
    if preds == 1:
        sentiment = 'positive'

    return preds, sentiment


In [51]:
# try a positive sentence
sentence = "It's such a nice day, think i'll be taking Sid to Ramsgate fish and chips for lunch at Peter's fish factory and then the beach maybe"
tmp_pred, tmp_sentiment = predict(sentence)
print(f"The sentiment of the sentence \n***\n\"{sentence}\"\n***\nis {tmp_sentiment}.")

print()
# try a negative sentence
sentence = "I hated my day, it was the worst, I'm so sad."
tmp_pred, tmp_sentiment = predict(sentence)
print(f"The sentiment of the sentence \n***\n\"{sentence}\"\n***\nis {tmp_sentiment}.")

Error in cell 106: name 'process_tweet' is not defined


Traceback (most recent call last):
  File "C:\Users\admin\.gemini\antigravity-ide\brain\cf7abe91-ac96-4e61-b868-3c92652c937f\scratch\single_nb_output_saver.py", line 71, in <module>
    exec(clean_source, global_env)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 3, in <module>
  File "<string>", line 3, in predict
  File "<string>", line 4, in tweet_to_tensor
NameError: name 'process_tweet' is not defined


Notice that the model works well even for complex sentences.

### On Deep Nets

Deep nets allow you to understand and capture dependencies that you would have not been able to capture with a simple linear regression, or logistic regression. 
- It also allows you to better use pre-trained embeddings for classification and tends to generalize better.